In [1]:
# ================================
# 1️⃣ Install Dependencies
# ================================
!pip install -q transformers datasets scikit-learn tqdm

# ================================
# 2️⃣ Imports
# ================================
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel, get_scheduler
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm

# Mixed precision (UPDATED API)
from torch.amp import autocast, GradScaler

# ================================
# 3️⃣ Device Setup
# ================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ================================
# 4️⃣ Load Dataset
# ================================
file_path = "/content/CN_relations_fixed.json"

with open(file_path, "r") as f:
    data = json.load(f)

dataset = Dataset.from_list(data)

# ================================
# 5️⃣ Labels
# ================================
LABELS = sorted(list(set([ex["output"].strip() for ex in data])))
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

def encode_labels(example):
    example["label"] = label2id[example["output"].strip()]
    return example

dataset = dataset.map(encode_labels)

# ================================
# 6️⃣ Tokenizer
# ================================
model_name = "allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["input"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

dataset = dataset.map(tokenize, batched=True, batch_size=1000)
dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"],
    output_all_columns=True
)

# ================================
# 7️⃣ Train / Val Split
# ================================
train_size = int(0.8 * len(dataset))
train_dataset = dataset.select(range(train_size))
val_dataset = dataset.select(range(train_size, len(dataset)))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, num_workers=2, pin_memory=True)

# ================================
# 8️⃣ Model
# ================================
base_model = AutoModel.from_pretrained(model_name)

class SciBERTClassifier(nn.Module):
    def __init__(self, base_model, num_labels):
        super().__init__()
        self.bert = base_model
        self.classifier = nn.Linear(base_model.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_output)

model = SciBERTClassifier(base_model, len(LABELS)).to(device)

# OPTIONAL: Freeze backbone
# for param in model.bert.parameters():
#     param.requires_grad = False

# ================================
# 9️⃣ Optimizer & Scheduler
# ================================
optimizer = AdamW(model.parameters(), lr=2e-5)

num_epochs = 3
num_training_steps = num_epochs * len(train_loader)

scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)

criterion = nn.CrossEntropyLoss()
scaler = GradScaler("cuda")

# ================================
# 🔟 Training Loop
# ================================
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader)

    for batch in loop:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad()

        with autocast("cuda"):
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()
        loop.set_description(f"Epoch {epoch+1}")
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} Avg Loss: {total_loss/len(train_loader):.4f}")

import re

import re

# ================================
# 1️⃣1️⃣ Evaluation + Predictions (FINAL FIXED)
# ================================
model.eval()
y_true, y_pred = [], []
results = []

val_idx = 0

with torch.no_grad():
    for batch in tqdm(val_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

        for i in range(len(labels)):
            ex = val_dataset[val_idx]
            text = ex["input"]

            # ✅ Extract entity name + label (FIXED INDENT)
            e1 = re.search(r'Entity1:\s*(.*?)\s*\((.*?)\)', text)
            e2 = re.search(r'Entity2:\s*(.*?)\s*\((.*?)\)', text)

            entity1 = e1.group(1).strip() if e1 else ""
            entity1_type = e1.group(2).strip() if e1 else ""

            entity2 = e2.group(1).strip() if e2 else ""
            entity2_type = e2.group(2).strip() if e2 else ""

            # ✅ Save WITH TYPES
            results.append({
                "entity1": entity1,
                "entity1_type": entity1_type,
                "relation": id2label[preds[i].item()],
                "entity2": entity2,
                "entity2_type": entity2_type
            })

            val_idx += 1

# ================================
# 1️⃣2️⃣ Metrics
# ================================
accuracy = accuracy_score(y_true, y_pred)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

print("\n===== RESULTS =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

# ================================
# 1️⃣3️⃣ Save + Download Predictions
# ================================
output_file = "/content/scibert-finetune_predictions.json"

with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

print(f"\n✅ Predictions saved to: {output_file}")

from google.colab import files
files.download(output_file)

Using device: cuda


Map:   0%|          | 0/1952 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Epoch 3 Avg Loss: 0.6397


100%|██████████| 13/13 [00:02<00:00,  4.74it/s]


===== RESULTS =====
Accuracy : 0.8031
Precision: 0.7718
Recall   : 0.8031
F1 Score : 0.7864

✅ Predictions saved to: /content/scibert-finetune_predictions.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>